In [56]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

/var/folders/wk/tl2drvdn73s_lt_qcp_4ksx40000gn/T/ipykernel_23576/1055050788.py:14: DtypeWarning:

Columns (8,25) have mixed types. Specify dtype option on import or set low_memory=False.



In [11]:
final_df.head()
final_df.columns
print(final_df.shape)
print(final_df[['id', 'name']].head())



(159463, 81)
         id                       name
0  419296.0                     Gotcha
1   18080.0                  Half-Life
2    5159.0     Resident Evil 2 (1998)
3   19299.0                  Fallout 2
4    5426.0  Crash Bandicoot 3: Warped


In [ ]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import plotly.express as px
import plotly.graph_objects as go
import socket
import pandas as pd

# =========================
# App Setup
# =========================

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP], suppress_callback_exceptions=True)
app.title = "Dashboard"
years = list(range(1970, 2025))

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()

# =========================
# Component Builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Companies", id="companies-button", n_clicks=0, color="primary" if active_tab == "companies" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(id="search_bar", placeholder="Search...", type="text", debounce=False),
            html.Div(id="search_results", style={
                "position": "absolute", "zIndex": 1000, "width": "100%", "backgroundColor": "white",
                "border": "1px solid #ced4da", "borderRadius": "0.25rem", "boxShadow": "0 2px 6px rgba(0,0,0,0.2)",
                "maxHeight": "200px", "overflowY": "auto", "marginTop": "2px"
            })
        ]),
        style={"position": "relative", "marginTop": "1rem", "marginBottom": "1rem", "backgroundColor": "white"}
    )

def build_games_middle(selected_sub, selected_sort_options):
    options = ["Most Popular"]
    buttons = [
        dbc.Button(
            label,
            id={"type": "sub-button", "index": label.lower().replace(" ", "-") + "-sub"},
            color="primary" if selected_sub == label.lower().replace(" ", "-") + "-sub" else "secondary",
            n_clicks=0,
            style={"width": "100%", "marginBottom": "0.5rem"}
        )
        for label in options
    ]

    sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
    sort_buttons = [
        dbc.Button(
            m,
            id={"type": "sort-button", "index": m},
            color="primary" if m in selected_sort_options else "secondary",
            style={"width": "100%", "marginBottom": "0.25rem"}
        )
        for m in sort_options
    ]

    return html.Div([
        dbc.Row([
            dbc.Col(buttons, width=6),
            dbc.Col([
                html.H6("Sort By:"),
                *sort_buttons
            ], width=6)
        ]),

        # ── Release-year slider (unchanged) ────────────────────────────────
        html.Div([
            html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970,
                max=2024,
                step=1,
                marks={y: str(y) for y in range(1970, 2025, 10)},
                value=[2000, 2020],
                tooltip={"placement": "bottom", "always_visible": False},
                allowCross=False,
                updatemode="mouseup"
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        }),

        # ── NEW: number-of-games slider ────────────────────────────────────
        html.Div([
            html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Slider(
                id="num-games-slider",
                min=3, max=200, step=1, value=50,                 # default 50
                marks={i: str(i) for i in range(10, 201, 30)},
                updatemode="drag", tooltip={"placement": "bottom", "always_visible": False}
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        })
    ])


def build_sidebar(active_tab, selected_sub, selected_sort_options):
    return html.Div([
        build_search_bar(),
        html.Hr(),
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=build_games_middle(selected_sub, selected_sort_options),
            style={"paddingTop": "1rem", "paddingBottom": "1rem"}
        )
    ], style={"padding": "1rem"})

def build_data_view():
    return html.Div([
        html.Div(id="main_graph"),
        html.Hr(),
        html.Div(id="comparison_panel")
    ])

# =========================
# App Layout
# =========================

initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"

app.layout = dbc.Container([
    dcc.Store(id="active_main_tab", data=initial_active_tab),
    dcc.Store(id="selected_sub_button", data=initial_selected_sub),
    dcc.Store(id="selected_sort_options", data=[]),
    dcc.Store(id="selected_year_range", data=[2000, 2020]),
    dcc.Store(id="selected_game_ids", data=[]),
    dcc.Store(id="last_clicked_timestamp"),

    dbc.Row([
        dbc.Col(id="sidebar", children=build_sidebar(initial_active_tab, initial_selected_sub, []), width=3,
                style={"backgroundColor": "#f8f9fa", "height": "100vh", "padding": 0,
                       "borderRight": "1px solid #dee2e6", "display": "flex", "flexDirection": "column"}),
        dbc.Col(build_data_view(), width=9)
    ])
], fluid=True)

# =========================
# Callbacks
# =========================

@app.callback(
    Output("search_results", "children"),
    Input("search_bar", "value"),
    prevent_initial_call=True
)
def update_search_results(search_text):
    if not search_text or len(search_text.strip()) < 2:
        return ""

    mask = final_df["name"].str.contains(search_text, case=False, na=False)
    matches = final_df.loc[mask].head(10)

    if matches.empty:
        return html.Small("No matches", style={"color": "#888"})

    return dbc.ListGroup([
        dbc.ListGroupItem(
            html.Span(
                row["name"],
                id={"type": "game-link", "index": int(row["id"])},
                n_clicks=0,
                style={"color": "#0d6efd", "cursor": "pointer", "textDecoration": "underline"}
            ),
            style={"padding": "0.4rem 0.6rem"}
        )
        for _, row in matches.iterrows()
    ], flush=True)

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    sort_option = triggered["index"]
    return [sort_option]


@app.callback(
    Output("comparison_panel", "children"),
    Input("selected_game_ids", "data"),
    prevent_initial_call=True
)
def update_comparison_panel(game_ids):
    if not game_ids:
        return html.Div("") # panel description effjes leeg

    cards = []
    for game_id in game_ids:
        match = final_df[final_df["id"] == game_id]
        if match.empty:
            continue
        row = match.iloc[0]

        card = dbc.Col([
            dbc.Card([
                dbc.CardHeader(
                    html.Div([
                        html.Span(row.get("name", "Unnamed Game"), style={"fontWeight": "bold", "fontSize": "1.2rem"}),
                        html.Button("×", id={"type": "remove-game", "index": game_id},
                                    style={
                                        "float": "right", "border": "none", "background": "none",
                                        "fontSize": "20px", "cursor": "pointer", "color": "red"
                                    })
                    ])
                ),
                html.Div([
                    html.Div([
                        html.P(f"Rating: {row.get('rating', 'N/A')}"),
                        html.P(f"Metacritic: {row.get('metacritic', 'N/A')}"),
                        html.P(f"Released: {row.get('released', 'Unknown')}"),
                        html.P(f"Added by users: {row.get('added', 'N/A')}"),
                        html.P(f"YouTube count: {row.get('youtube_count', 'N/A')}"),
                        html.P(f"Twitch count: {row.get('twitch_count', 'N/A')}"),
                        html.P(
                            (row.get("description_raw") or "")[:180] + "..."
                        ) if isinstance(row.get("description_raw"), str) else html.P("No description available.")
                    ], style={
                        "position": "absolute",
                        "top": 0,
                        "left": 0,
                        "right": 0,
                        "bottom": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "padding": "0.75rem",
                        "borderRadius": "0 0 8px 8px",
                        "color": "white"
                    })
                ], style={
                    "position": "relative",
                    "backgroundImage": f"url('{row.get('background_image', '')}')",
                    "backgroundSize": "cover",
                    "backgroundPosition": "center",
                    "borderRadius": "8px",
                    "height": "240px",
                    "overflow": "hidden"
                })
            ], style={"margin": "0.5rem", "height": "300px", "overflow": "hidden"})
        ], width=6)
        cards.append(card)

    return dbc.Row(cards)




@app.callback(
    Output("selected_game_ids", "data"),
    Output("last_clicked_timestamp", "data"),
    Input({"type": "game-link", "index": ALL}, "n_clicks_timestamp"),
    Input({"type": "remove-game", "index": ALL}, "n_clicks"),
    State({"type": "game-link", "index": ALL}, "id"),
    State("selected_game_ids", "data"),
    State("last_clicked_timestamp", "data"),
    prevent_initial_call=True
)
def handle_game_selection_and_removal(n_clicks_ts, remove_clicks, ids, selected_ids, last_ts):
    if selected_ids is None:
        selected_ids = []

    triggered = ctx.triggered_id

    # Handle removal
    if isinstance(triggered, dict) and triggered.get("type") == "remove-game":
        removed_id = triggered["index"]
        return [gid for gid in selected_ids if gid != removed_id], last_ts

    # Handle selection
    max_ts = -1
    clicked_id = None

    for ts, btn_id in zip(n_clicks_ts, ids):
        if ts is not None and ts > max_ts:
            max_ts = ts
            clicked_id = btn_id["index"]

    if clicked_id is not None and (last_ts is None or max_ts > last_ts):
        if clicked_id in selected_ids:
            return selected_ids, max_ts

        if len(selected_ids) == 2:
            selected_ids.pop(0)

        selected_ids.append(clicked_id)
        return selected_ids, max_ts

    return dash.no_update, dash.no_update



# =========================
# MAIN GRAPH – HEAT-MAP VERSION
# =========================

import plotly.express as px
import squarify
import numpy as np

@app.callback(
    Output("main_graph", "children"),
    Input("selected_sort_options", "data"),
    Input("selected_year_range", "data"),
    Input("num-games-slider", "value")
)
def update_main_graph(selected_sort_options, selected_year_range, num_games):
    # 1. Filter & rank
    df = final_df.copy()
    df = df.dropna(subset=["name"]).drop_duplicates()
    df = df[
        (df["release_year"] >= selected_year_range[0]) &
        (df["release_year"] <= selected_year_range[1])
    ]

    column_mapping = {
        "rating": "rating",
        "youtube": "youtube_count",
        "twitch": "twitch_count",
        "added": "added",
        "metacritic": "metacritic",
    }
    sort_by = column_mapping.get(
        (selected_sort_options[0] if selected_sort_options else "rating").lower(),
        "rating",
    )

    num_games = num_games or 50           # fallback default
    df = (
        df.dropna(subset=[sort_by])
          .sort_values(sort_by, ascending=False)
          .head(num_games)
    )

    # 2. Treemap areas – **linear** with the metric
    #    Guard against zeros so we don't hand 0-area to squarify.
    metric = df[sort_by].astype(float).clip(lower=1e-6)
    areas  = metric                       # <- direct proportionality
    normed = squarify.normalize_sizes(areas, 100, 100)
    rects  = squarify.squarify(normed, 0, 0, 100, 100)
    df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

    # 3. Build tiles (unchanged visual style)
    tiles = [
        html.Div(
            children=[
                html.Div(
                    [
                        html.Div(
                            row["name"],
                            style={
                                "fontSize": "12px", "fontWeight": "bold",
                                "overflow": "hidden", "textOverflow": "ellipsis",
                                "whiteSpace": "nowrap",
                            },
                        ),
                        html.Div(
                            f"{sort_by.capitalize()}: {row[sort_by]:.2f}",
                            style={"fontSize": "10px"},
                        ),
                    ],
                    style={
                        "position": "absolute", "inset": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "display": "flex", "flexDirection": "column",
                        "alignItems": "center", "justifyContent": "center",
                        "padding": "4px",
                    },
                )
            ],
            style={
                "position": "absolute",
                "left":   f"{row['x']:.2f}%",  "top":    f"{row['y']:.2f}%",
                "width":  f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                "backgroundImage":  f"url('{row['background_image']}')",
                "backgroundSize":   "cover",
                "backgroundPosition": "center",
                "border": "1px solid #fff",
                "boxSizing": "border-box",
                "borderRadius": "4px",
                "overflow": "hidden",
                "color": "#fff",
            },
        )
        for _, row in df.iterrows()
    ]

    # 4. Return wrapped container
    return html.Div(
        tiles,
        style={
            "position": "relative",
            "width": "100%",
            "height": "60vh",
            "backgroundColor": "#333",
        },
    )


@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new(f"http://127.0.0.1:{PORT}")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, port=PORT)
